# NB02 — ViSoBERT + Residual DS-CNN + Adaptive Gated Fusion (v3)

Ablation công bằng giữa `concat` và `gated`: cùng pretrained encoder, residual depthwise–pointwise CNN đa tỉ lệ, loss, seed và training budget; chỉ khác cơ chế fusion. Kết quả chính dùng raw argmax của best-validation checkpoint, không hiệu chỉnh theo test.

> V3 được thiết kế sau khi đã quan sát test v2, vì vậy test ViHSD không còn là holdout hoàn toàn mù cho quyết định kiến trúc. Cần công bố điều này và xác nhận bằng nhiều seed/holdout mới trước khi tuyên bố SOTA.

In [ ]:
# USER SETTINGS — this must be the first code cell
MODEL_VARIANTS = ('concat', 'gated')
SEED = 42
EPOCHS = 10.0
EVALUATE_TEST = True
MAX_LENGTH, MICRO_BATCH, GRAD_ACCUMULATION, EVAL_BATCH = 128, 8, 5, 16
CNN_DIM, CNN_KERNELS, FUSION_DIM = 192, (3, 5, 7), 256
CNN_DROPOUT, HEAD_DROPOUT = 0.10, 0.20
ENCODER_LR, HEAD_LR, WEIGHT_DECAY, WARMUP_RATIO = 1.2e-5, 4e-5, 0.01, 0.10
FOCAL_GAMMA, EFFECTIVE_NUMBER_BETA, BOOTSTRAP_SAMPLES = 1.0, 0.9999, 2000

import os, sys, subprocess
from datetime import datetime, timezone
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert MODEL_VARIANTS == ('concat', 'gated') and MICRO_BATCH * GRAD_ACCUMULATION == 40
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'datasets', 'accelerate', 'sentencepiece', 'protobuf',
    'huggingface_hub', 'scikit-learn', 'pandas', 'pyarrow', 'matplotlib', 'safetensors'], check=True)
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ROOT_RUN_DIR = Path('/kaggle/working') / f'proposed_dscnn_gate_v3_seed{SEED}_{RUN_TAG}'
ROOT_RUN_DIR.mkdir(parents=True, exist_ok=False)
print({'root_run_dir': str(ROOT_RUN_DIR), 'effective_batch': 40})

In [ ]:
import hashlib, json, random, gc
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, set_seed

assert torch.cuda.is_available() and torch.cuda.device_count() == 1
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_ACCOUNT = HfApi(token=HF_TOKEN).whoami()['name']
except Exception as exc:
    raise RuntimeError('Enable a valid Kaggle Secret named HF_TOKEN.') from exc
manifest_paths = list(Path('/kaggle/input').rglob('vihsd_manifest.json'))
if len(manifest_paths) != 1: raise RuntimeError(f'Attach exactly one prepared output: {manifest_paths}')
manifest_path = manifest_paths[0]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['artifact'] == 'vihsd_prepared_v1'

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

def load_verified_frame(split):
    spec = manifest['splits'][split]; path = manifest_path.parent / spec['file']
    if sha256(path) != spec['sha256']: raise ValueError(f'{split} hash mismatch')
    frame = pd.read_parquet(path)
    if len(frame) != spec['rows']: raise ValueError(f'{split} row mismatch')
    return frame

MODEL_ID, MODEL_SHA = manifest['model_id'], manifest['model_revision']
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN)
tokenizer.padding_side = 'right'
def make_dataset(frame):
    ds = Dataset.from_dict({'text': frame.text.fillna('').astype(str).tolist(),
                            'labels': frame.labels.astype(int).tolist()})
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=['text'])

# Test stays unopened until both models and all hyperparameters are locked.
train_frame, validation_frame = load_verified_frame('train'), load_verified_frame('validation')
train_ds, validation_ds = make_dataset(train_frame), make_dataset(validation_frame)
train_counts = np.bincount(train_frame.labels.astype(int), minlength=3)
if np.any(train_counts == 0): raise ValueError('All three classes must occur in train.')
effective = 1.0 - EFFECTIVE_NUMBER_BETA ** train_counts.astype(np.float64)
class_weights = ((1.0 - EFFECTIVE_NUMBER_BETA) / np.maximum(effective, 1e-12))
class_weights /= class_weights.mean()
print({'gpu': torch.cuda.get_device_name(0), 'hf_account': HF_ACCOUNT,
       'train_counts': train_counts.tolist(), 'class_weights': class_weights.round(4).tolist(),
       'model_revision': MODEL_SHA, 'dataset_revision': manifest['dataset_revision']})

## V3 architecture

- Global: masked mean + `<s>` → projection 256.
- Local: 768→192 pointwise projection → residual depthwise/pointwise kernels 3/5/7 → masked max+mean → projection 256.
- Gate: $a=\sigma(MLP[g,l,|g-l|,g\odot l])$, $z=LN(g+\sigma(s)a\odot(l-g))$. Gate output starts at 0.5 and $\sigma(s)\approx0.12$, so optimization begins near the stable Transformer path instead of amplifying both branches.

In [ ]:
from torch import nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
LABEL2ID = {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

class ProposedConfig(PretrainedConfig):
    model_type, is_composition = 'visobert-residual-dscnn-gate-v3', True
    def __init__(self, encoder_config=None, encoder_model_id='uitnlp/visobert',
                 encoder_revision=None, variant='gated', cnn_dim=192, cnn_kernels=(3,5,7),
                 fusion_dim=256, head_dropout=.2, cnn_dropout=.1, class_weights=None,
                 focal_gamma=1., num_labels=3, **kwargs):
        if variant not in ('concat', 'gated'): raise ValueError(f'Bad variant: {variant}')
        if any(int(k) < 1 or int(k) % 2 == 0 for k in cnn_kernels):
            raise ValueError('CNN kernels must be positive and odd.')
        super().__init__(num_labels=num_labels, **kwargs)
        self.encoder_config, self.encoder_model_id = dict(encoder_config or {}), encoder_model_id
        self.encoder_revision, self.variant = encoder_revision, variant
        self.cnn_dim, self.cnn_kernels, self.fusion_dim = int(cnn_dim), list(cnn_kernels), int(fusion_dim)
        self.head_dropout, self.cnn_dropout = float(head_dropout), float(cnn_dropout)
        self.class_weights, self.focal_gamma = list(class_weights or []), float(focal_gamma)

class ResidualDsConv(nn.Module):
    def __init__(self, channels, kernel, dropout):
        super().__init__()
        self.dw = nn.Conv1d(channels, channels, kernel, padding=kernel//2, groups=channels, bias=False)
        self.pw = nn.Conv1d(channels, channels, 1)
        self.dropout, self.norm = nn.Dropout(dropout), nn.LayerNorm(channels)
    def forward(self, tokens, mask):
        update = self.pw(F.gelu(self.dw(tokens.transpose(1,2)))).transpose(1,2)
        return self.norm(tokens + self.dropout(update)) * mask.unsqueeze(-1).to(tokens.dtype)

class MultiScaleDsCnnPool(nn.Module):
    def __init__(self, hidden, channels, kernels, dropout):
        super().__init__()
        self.proj, self.norm = nn.Linear(hidden, channels), nn.LayerNorm(channels)
        self.branches = nn.ModuleList([ResidualDsConv(channels, k, dropout) for k in kernels])
        self.output_size = 2 * channels * len(kernels)
    def forward(self, hidden, attention_mask):
        if hidden.shape[:2] != attention_mask.shape: raise ValueError('Mask shape mismatch')
        mask = attention_mask.bool()
        if not bool(mask.any(1).all()): raise ValueError('Empty sample')
        tokens = self.norm(F.gelu(self.proj(hidden))) * mask.unsqueeze(-1).to(hidden.dtype)
        pooled = []
        for branch in self.branches:
            x = branch(tokens, mask); valid = mask.unsqueeze(-1).to(x.dtype)
            maximum = x.masked_fill(~mask.unsqueeze(-1), torch.finfo(x.dtype).min).amax(1)
            mean = (x * valid).sum(1) / valid.sum(1).clamp_min(1)
            pooled.extend((maximum, mean))
        return torch.cat(pooled, -1)

class ProposedViSoBERT(PreTrainedModel):
    config_class, base_model_prefix, main_input_name = ProposedConfig, 'encoder', 'input_ids'
    accepts_loss_kwargs = False
    def __init__(self, config):
        super().__init__(config)
        payload = dict(config.encoder_config); model_type = payload.pop('model_type', None)
        if not model_type: raise ValueError('encoder_config.model_type is required')
        enc_cfg = AutoConfig.for_model(model_type, **payload)
        self.encoder = AutoModel.from_config(enc_cfg, add_pooling_layer=False)
        hidden = int(enc_cfg.hidden_size)
        self.global_proj, self.global_norm = nn.Linear(2*hidden, config.fusion_dim), nn.LayerNorm(config.fusion_dim)
        self.local_pool = MultiScaleDsCnnPool(hidden, config.cnn_dim, config.cnn_kernels, config.cnn_dropout)
        self.local_proj = nn.Linear(self.local_pool.output_size, config.fusion_dim)
        self.local_norm = nn.LayerNorm(config.fusion_dim)
        if config.variant == 'gated':
            self.gate_hidden = nn.Linear(4*config.fusion_dim, config.fusion_dim)
            self.gate_output = nn.Linear(config.fusion_dim, config.fusion_dim)
            self.residual_scale_logit = nn.Parameter(torch.tensor(-2.0))
            self.fusion_norm = nn.LayerNorm(config.fusion_dim)
            classifier_input = config.fusion_dim
        else: classifier_input = 2 * config.fusion_dim
        self.dropout, self.classifier = nn.Dropout(config.head_dropout), nn.Linear(classifier_input, config.num_labels)
        self.register_buffer('class_weight_tensor', torch.tensor(config.class_weights, dtype=torch.float32), persistent=False)
        self.post_init()
        if config.variant == 'gated':
            nn.init.zeros_(self.gate_output.weight); nn.init.zeros_(self.gate_output.bias)
    def fuse(self, g, l):
        if self.config.variant == 'concat': return torch.cat([g, l], -1)
        features = torch.cat([g, l, torch.abs(g-l), g*l], -1)
        gate = torch.sigmoid(self.gate_output(F.gelu(self.gate_hidden(features))))
        return self.fusion_norm(g + torch.sigmoid(self.residual_scale_logit) * gate * (l-g))
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if attention_mask is None: raise ValueError('attention_mask is required')
        kwargs.pop('num_items_in_batch', None)
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, **kwargs)
        hidden = out.last_hidden_state; mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
        g = self.global_norm(F.gelu(self.global_proj(torch.cat([hidden[:,0], mean], -1))))
        l = self.local_norm(F.gelu(self.local_proj(self.local_pool(hidden, attention_mask))))
        logits = self.classifier(self.dropout(self.fuse(g, l))); loss = None
        if labels is not None:
            ce = F.cross_entropy(logits, labels, weight=self.class_weight_tensor.to(logits.dtype), reduction='none')
            p = logits.softmax(-1).gather(1, labels[:,None]).squeeze(1)
            loss = (((1-p).clamp_min(0) ** self.config.focal_gamma) * ce).mean()
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=out.hidden_states, attentions=out.attentions)

ARCHITECTURE_NAMES = {'concat': 'ViSoBERT + Residual DS-CNN Concat v3',
                      'gated': 'ViSoBERT + Residual DS-CNN Adaptive Gate v3'}
encoder_config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN)
def build_fresh_model(variant):
    cfg = ProposedConfig(encoder_config=encoder_config.to_dict(), encoder_model_id=MODEL_ID,
        encoder_revision=MODEL_SHA, variant=variant, cnn_dim=CNN_DIM, cnn_kernels=CNN_KERNELS,
        fusion_dim=FUSION_DIM, head_dropout=HEAD_DROPOUT, cnn_dropout=CNN_DROPOUT,
        class_weights=class_weights.tolist(), focal_gamma=FOCAL_GAMMA, num_labels=3,
        label2id=LABEL2ID, id2label=ID2LABEL)
    model = ProposedViSoBERT(cfg)
    pretrained = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN,
        add_pooling_layer=False, attn_implementation='eager')
    copied = model.encoder.load_state_dict(pretrained.state_dict(), strict=True); del pretrained
    if copied.missing_keys or copied.unexpected_keys: raise RuntimeError(copied)
    counts = {'total': sum(p.numel() for p in model.parameters()),
              'trainable': sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model, counts

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def metrics_from_logits(labels, logits):
    labels, pred = np.asarray(labels), np.asarray(logits).argmax(-1)
    _, recall, f1, support = precision_recall_fscore_support(labels, pred, labels=[0,1,2], zero_division=0)
    return {'accuracy': float(accuracy_score(labels, pred)), 'macro_f1': float(f1.mean()),
            'weighted_f1': float(np.average(f1, weights=support)),
            'per_class_f1': {ID2LABEL[i]: float(f1[i]) for i in range(3)},
            'per_class_recall': {ID2LABEL[i]: float(recall[i]) for i in range(3)}}
def compute_metrics(prediction):
    values = metrics_from_logits(prediction.label_ids, prediction.predictions)
    return {k:v for k,v in values.items() if isinstance(v, float)}
def optimizer_groups(model):
    groups = []
    for is_encoder, lr in ((True, ENCODER_LR), (False, HEAD_LR)):
        for decay in (True, False):
            params = [p for n,p in model.named_parameters()
                      if n.startswith('encoder.') == is_encoder and (p.ndim > 1) == decay]
            if params: groups.append({'params': params, 'lr': lr, 'weight_decay': WEIGHT_DECAY if decay else 0.})
    return groups

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
def train_and_validate(variant):
    random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
    run_dir = ROOT_RUN_DIR / variant; run_dir.mkdir(parents=True, exist_ok=False)
    model, parameters = build_fresh_model(variant)
    print({'phase':'train', 'model_variant':variant, 'architecture':ARCHITECTURE_NAMES[variant], 'parameters':parameters})
    args = TrainingArguments(output_dir=str(run_dir/'checkpoints'), num_train_epochs=EPOCHS,
        learning_rate=ENCODER_LR, per_device_train_batch_size=MICRO_BATCH,
        per_device_eval_batch_size=EVAL_BATCH, gradient_accumulation_steps=GRAD_ACCUMULATION,
        optim='adamw_torch', weight_decay=WEIGHT_DECAY, lr_scheduler_type='linear',
        warmup_ratio=WARMUP_RATIO, max_grad_norm=1., eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        save_total_limit=2, save_safetensors=True, fp16=True, seed=SEED, data_seed=SEED,
        report_to='none', logging_steps=50, dataloader_num_workers=0)
    optimizer = torch.optim.AdamW(optimizer_groups(model), betas=(.9,.999), eps=1e-8)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=validation_ds,
        processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
        optimizers=(optimizer, None))
    trainer.train()
    trainer.save_model(run_dir/'best_model'); tokenizer.save_pretrained(run_dir/'best_model')
    output = trainer.predict(validation_ds)
    metrics = metrics_from_logits(output.label_ids, output.predictions)
    result = {'experiment':f'PROPOSED_{variant.upper()}_V3', 'variant':variant,
        'architecture':ARCHITECTURE_NAMES[variant], 'validation_metrics':metrics,
        'best_checkpoint':trainer.state.best_model_checkpoint, 'parameters':parameters}
    (run_dir/'validation_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print({'phase':'validation', 'model_variant':variant, **metrics})
    return {'run_dir':run_dir, 'trainer':trainer, **result}

contexts = []
for variant in MODEL_VARIANTS:
    contexts.append(train_and_validate(variant))
    torch.cuda.empty_cache()
validation_table = pd.DataFrame([{'variant':c['variant'],
    'validation_macro_f1':c['validation_metrics']['macro_f1'], 'parameters':c['parameters']['total']}
    for c in contexts]).sort_values('validation_macro_f1', ascending=False)
display(validation_table); validation_table.to_csv(ROOT_RUN_DIR/'validation_comparison.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

def stratified_ci(truth, predicted):
    rng = np.random.default_rng(SEED + 1000); groups = [np.flatnonzero(truth == i) for i in range(3)]
    scores = []
    for _ in range(BOOTSTRAP_SAMPLES):
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        scores.append(f1_score(truth[idx], predicted[idx], labels=[0,1,2], average='macro', zero_division=0))
    return [float(v) for v in np.quantile(scores, [.025,.975])]

assert EVALUATE_TEST is True
test_frame = load_verified_frame('test')
test_ds = make_dataset(test_frame)
def test_and_save(context):
    variant, trainer, run_dir = context['variant'], context['trainer'], context['run_dir']
    print({'phase':'test', 'model_variant':variant})
    output = trainer.predict(test_ds); truth = np.asarray(output.label_ids)
    logits = np.asarray(output.predictions); pred = logits.argmax(-1)
    probs = torch.softmax(torch.from_numpy(logits), -1).numpy()
    metrics = metrics_from_logits(truth, logits); metrics['macro_f1_bootstrap_95ci'] = stratified_ci(truth, pred)
    print(json.dumps({'model_variant':variant, 'split':'test',
        'decision_rule':'raw_argmax_no_test_tuning', 'metrics':metrics}, indent=2))
    display(pd.DataFrame(classification_report(truth, pred, labels=[0,1,2],
        target_names=['CLEAN','OFFENSIVE','HATE'], zero_division=0, output_dict=True)).T)
    matrix = confusion_matrix(truth, pred, labels=[0,1,2]); fig, ax = plt.subplots(figsize=(5,4))
    ax.imshow(matrix, cmap='Purples')
    for r in range(3):
        for c in range(3): ax.text(c, r, int(matrix[r,c]), ha='center', va='center')
    ax.set_xticks(range(3), ['CLEAN','OFFENSIVE','HATE']); ax.set_yticks(range(3), ['CLEAN','OFFENSIVE','HATE'])
    ax.set(xlabel='Predicted', ylabel='True', title=f'{variant.upper()} v3 — test'); plt.show()
    pd.DataFrame({'y_true':truth, 'y_pred':pred, 'p_clean':probs[:,0],
        'p_offensive':probs[:,1], 'p_hate':probs[:,2]}).to_parquet(run_dir/'predictions_test.parquet', index=False)
    result = {'experiment':context['experiment'], 'model_variant':variant,
        'architecture':context['architecture'], 'seed':SEED, 'selection_split':'validation',
        'test_decision_rule':'raw_argmax', 'test_informed_redesign_disclosure':True,
        'model_revision':MODEL_SHA, 'dataset_revision':manifest['dataset_revision'],
        'validation_metrics':context['validation_metrics'], 'test_metrics':metrics,
        'parameters':context['parameters']}
    (run_dir/'test_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result

all_results = []
for context in contexts:
    all_results.append(test_and_save(context))
    del context['trainer']; gc.collect(); torch.cuda.empty_cache()
comparison = pd.DataFrame([{'variant':x['model_variant'],
    'validation_macro_f1':x['validation_metrics']['macro_f1'],
    'test_macro_f1':x['test_metrics']['macro_f1'],
    'test_ci_low':x['test_metrics']['macro_f1_bootstrap_95ci'][0],
    'test_ci_high':x['test_metrics']['macro_f1_bootstrap_95ci'][1],
    'test_accuracy':x['test_metrics']['accuracy'], 'parameters':x['parameters']['total']} for x in all_results])
display(comparison); comparison.to_csv(ROOT_RUN_DIR/'comparison.csv', index=False)
(ROOT_RUN_DIR/'all_results.json').write_text(json.dumps(all_results, indent=2), encoding='utf-8')
print(f'SUCCESS: {ROOT_RUN_DIR}')
print('For a paper claim, repeat unchanged with seeds 42/52/62 and report mean ± std.')